# Execute one frozen v3 stage

**Inputs:** `AHRV3_STAGE_INDEX` and immutable protocol. **Outputs:** one named training/evaluation/diagnostic/handoff stage. **Default Run All:** definition import and inspection only. **Runtime:** stage-dependent, seconds to hours; long runs are child processes of the detached controller. **Exact resume:** use `protocol_and_run.ipynb` in run mode; never manually launch a concurrent child. Only tagged upstream definitions are imported.

In [ ]:
from pathlib import Path
import nbformat,os
PROJECT=Path.cwd().resolve()
while not (PROJECT/'notebooks/00_instance_setup_and_recovery.ipynb').is_file():
    if PROJECT.parent==PROJECT:raise RuntimeError('Run within AHR Go workspace')
    PROJECT=PROJECT.parent
for cell in nbformat.read(PROJECT/'experiments/ahr_jepa_v3/00_shared.ipynb',4).cells:
    if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):exec(compile(cell.source,'v3-shared:export','exec'),globals())
v3_load_upstream(PROJECT,globals())
for name in ('08_predictive_ahr.ipynb','09_trajectory_learning.ipynb','11_controlled_evaluation.ipynb','13_local_handoff.ipynb','progress.ipynb'):
    v3_import_exports(PROJECT/V3_ROOT/name,globals())
_pipeline_namespace={'__name__':'__ahr_v3_pipeline__'}
for dependency in ('notebooks/00_instance_setup_and_recovery.ipynb','reports/09_13_training_pipeline.ipynb'):
    v3_import_exports(PROJECT/dependency,_pipeline_namespace)
# Keep the tested generic controller's implementation and isolated globals.
_v3_original_checkpoint_save=_pipeline_namespace['_save_checkpoint_unlocked']
def _v3_controller_checkpoint_save(*args,**kwargs):
    kwargs['resume_cell']='experiments/ahr_jepa_v3/protocol_and_run.ipynb: AHRV3_MODE=run; fresh kernel Run All'
    return _v3_original_checkpoint_save(*args,**kwargs)
_pipeline_namespace['_save_checkpoint_unlocked']=_v3_controller_checkpoint_save
v3_run_controller=_pipeline_namespace['run_training_pipeline']

v3_import_exports(PROJECT/V3_ROOT/'protocol_and_run.ipynb',globals())


In [ ]:
if 'AHRV3_STAGE_INDEX' not in os.environ:print('INSPECT_ONLY: the controller supplies a declared stage index.')
else:
    result=v3_dispatch_stage(PROJECT,int(os.environ['AHRV3_STAGE_INDEX']))
    print(json.dumps({'stage':int(os.environ['AHRV3_STAGE_INDEX']),'status':result.get('status','RECORDED')},indent=2))